In [80]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [81]:
##langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings

## vector store imports
from langchain_community.vectorstores import Chroma

import numpy as np
from typing import List




###  1 Sample Data Creation


In [82]:
sample_docs = [""""
Machine Learning
Machine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task. It uses algorithms and statistical techniques to identify patterns, relationships, and trends within large datasets. Machine Learning is generally divided into three major types: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labelled datasets to train models for tasks such as predicting house prices, detecting spam emails, and forecasting sales. Unsupervised learning identifies hidden patterns in unlabelled data through techniques such as clustering and dimensionality reduction. Reinforcement learning allows an agent to learn through interactions with an environment by receiving rewards or penalties for its actions. Popular Machine Learning algorithms include Linear Regression, Logistic Regression, Decision Trees, Random Forest, Support Vector Machines, and K-Nearest Neighbors. Machine Learning is widely used in healthcare, banking, recommendation systems, fraud detection, autonomous vehicles, and customer analytics. Its effectiveness depends on the quality of training data, appropriate feature selection, model evaluation, and continuous optimization.
""",
"""
Deep Learning
Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of data. Inspired by the structure and functioning of the human brain, deep learning models consist of interconnected artificial neurons organized into input, hidden, and output layers. Each layer processes information and extracts increasingly complex features, allowing the model to perform tasks that are difficult for traditional Machine Learning algorithms. Some of the most important architectures include Artificial Neural Networks (ANNs), Convolutional Neural Networks (CNNs), Recurrent Neural Networks (RNNs), Long Short-Term Memory networks (LSTMs), and Transformers. CNNs are particularly effective in image recognition, object detection, and medical image analysis, while RNNs and LSTMs are designed to process sequential information such as speech and time-series data. Transformers have revolutionized modern AI applications by enabling efficient processing of large amounts of sequential information. Deep Learning powers technologies such as facial recognition, autonomous driving, voice assistants, image generation, and large language models. However, deep learning generally requires substantial computational resources, large datasets, and careful model training to achieve high performance.
""",
"""
Natural Language Processing (NLP)
Natural Language Processing (NLP) is a branch of Artificial Intelligence that focuses on enabling computers to understand, interpret, process, and generate human language. It combines concepts from linguistics, computer science, Machine Learning, and Deep Learning to help machines communicate with humans using natural languages such as English, Hindi, and other languages. NLP involves several important techniques, including tokenization, stemming, lemmatization, part-of-speech tagging, named entity recognition, sentiment analysis, and text classification. Tokenization divides sentences into smaller units called tokens, while named entity recognition identifies important information such as names of people, organizations, and geographical locations. Sentiment analysis determines whether a given piece of text expresses positive, negative, or neutral opinions. Modern NLP systems also use advanced architectures such as Transformers, which power models like BERT and GPT. These models can perform complex tasks including language translation, text summarization, question answering, conversational AI, and content generation. NLP is widely used in chatbots, virtual assistants, search engines, customer support automation, document analysis, and social media monitoring. With the development of Large Language Models (LLMs), NLP has become one of the most rapidly evolving areas of Artificial Intelligence, making human-computer interaction more natural, intelligent, and accessible.
"""
]

sample_docs

['"\nMachine Learning\nMachine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task. It uses algorithms and statistical techniques to identify patterns, relationships, and trends within large datasets. Machine Learning is generally divided into three major types: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labelled datasets to train models for tasks such as predicting house prices, detecting spam emails, and forecasting sales. Unsupervised learning identifies hidden patterns in unlabelled data through techniques such as clustering and dimensionality reduction. Reinforcement learning allows an agent to learn through interactions with an environment by receiving rewards or penalties for its actions. Popular Machine Learning algorithms include Linear Regression, Logistic Regression, Decision Trees, Random Forest

In [83]:
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)

### 2. Document Loading 

In [84]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "data", 
    glob="*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents from the directory.")
print(f"First document content:\n{documents[0].page_content[:100]}...")  # Print first 500 characters of the first document


Loaded 3 documents from the directory.
First document content:
"
Machine Learning
Machine Learning (ML) is a branch of Artificial Intelligence that enables compute...


In [85]:
temp_dir


'C:\\Users\\DUNDES~1\\AppData\\Local\\Temp\\tmpigrkuatr'

### 3 Document Splitting

In [86]:
documents 

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='"\nMachine Learning\nMachine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task. It uses algorithms and statistical techniques to identify patterns, relationships, and trends within large datasets. Machine Learning is generally divided into three major types: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labelled datasets to train models for tasks such as predicting house prices, detecting spam emails, and forecasting sales. Unsupervised learning identifies hidden patterns in unlabelled data through techniques such as clustering and dimensionality reduction. Reinforcement learning allows an agent to learn through interactions with an environment by receiving rewards or penalties for its actions. Popular Machine Learning algorithms include Linear 

In [87]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Size of each chunk
    chunk_overlap=50,  # Overlap between chunks
    length_function=len,
    separators=[ " "] 
)

chunks = text_splitter.split_documents(documents)
print(f"Number of chunks created: {len(chunks)}")
print(f"First chunk content:\n{chunks[0].page_content[:100]}...")  # Print first 100 characters of the first chunk

Number of chunks created: 11
First chunk content:
"
Machine Learning
Machine Learning (ML) is a branch of Artificial Intelligence that enables compute...


### 4 Embedding Models

In [88]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-small-en-v1.5"
model_kwargs = {"device": "cpu"}

try:
    # Load from disk only if already present
    print(f"Searching for {model_name} locally...")
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name, model_kwargs={**model_kwargs, "local_files_only": True}
    )
    print("Model loaded from local storage.")

except Exception as e:
    # If not found, download it
    print("Model not found locally. Downloading from Hugging Face...")
    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
    print("Download complete and model initialized!")

Searching for BAAI/bge-small-en-v1.5 locally...
Model loaded from local storage.


In [89]:
vec = embeddings.embed_query("What is machine learning?")
print(len(vec))      # should print 384
print(vec[:5])       # a few floats like 0.03, -0.01, ...

384
[-0.0034288568422198296, -0.037564780563116074, -0.008593402802944183, -0.015035971999168396, 0.05639936402440071]


In [90]:
import numpy as np

def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

q  = embeddings.embed_query("How do computers learn from data?")
d1 = embeddings.embed_query("Machine learning lets computers learn patterns from data.")
d2 = embeddings.embed_query("My favourite pizza topping is mushrooms.")

print("related  :", cos(q, d1))
print("unrelated:", cos(q, d2))

related  : 0.8302973693917174
unrelated: 0.3023814490185264


### Initialize the chromaDb and store the chunks in the vector representation

In [91]:
persistence_dir = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding = HuggingFaceEmbeddings (
        model_name="BAAI/bge-small-en-v1.5", model_kwargs = {"device": "cpu"}
    ),
    persist_directory=persistence_dir,
    collection_name="rag_collection"
)

print(f"Vector store created and persisted at: {persistence_dir}")
print(f"Number of documents in the vector store: {vectorstore._collection.count()}")


Vector store created and persisted at: ./chroma_db
Number of documents in the vector store: 55


### Similarity Search


In [92]:
query = "What is NLP?"

similarity_search_results = vectorstore.similarity_search(query, k=3)
similarity_search_results

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\nNatural Language Processing (NLP) is a branch of Artificial Intelligence that focuses on enabling computers to understand, interpret, process, and generate human language. It combines concepts from linguistics, computer science, Machine Learning, and Deep Learning to help machines communicate with humans using natural languages such as English, Hindi, and other languages. NLP involves several important techniques, including tokenization, stemming,'),
 Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\nNatural Language Processing (NLP) is a branch of Artificial Intelligence that focuses on enabling computers to understand, interpret, process, and generate human language. It combines concepts from linguistics, computer science, Machine Learning, and Deep Learning to help machines communicate with humans using natural languages such as English, 

In [93]:
query = "What is deep learning ?"

similarity_search_results = vectorstore.similarity_search(query, k=3)
similarity_search_results

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of data. Inspired by the structure and functioning of the human brain, deep learning models consist of interconnected artificial neurons organized into input, hidden, and output layers. Each layer processes information and extracts increasingly complex features, allowing the model to perform tasks that'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of data. Inspired by the structure and functioning of the human brain, deep learning models consist of interconnected artificial neurons organized into input, hidden, and ou

In [94]:
print(f"Query '{query}':")
for i, result in enumerate(similarity_search_results):
    print(f"\n   Result {i+1}:")
    print(f"Content: {result.page_content[:200]}...")  # Print first 200 characters of the result
    print(f"Metadata: {result.metadata}")

Query 'What is deep learning ?':

   Result 1:
Content: Deep Learning
Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of da...
Metadata: {'source': 'data\\doc_1.txt'}

   Result 2:
Content: Deep Learning
Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of da...
Metadata: {'source': 'data\\doc_1.txt'}

   Result 3:
Content: Deep Learning
Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of da...
Metadata: {'source': 'data\\doc_1.txt'}


### Advanced Similarity Search with Scores

In [95]:
results_scores = vectorstore.similarity_search_with_score(query, k=3)
results_scores

[(Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of data. Inspired by the structure and functioning of the human brain, deep learning models consist of interconnected artificial neurons organized into input, hidden, and output layers. Each layer processes information and extracts increasingly complex features, allowing the model to perform tasks that'),
  0.22733156383037567),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning\nDeep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of data. Inspired by the structure and functioning of the human brain, deep learning models consist of interconnected artificial neurons organized 

### understanding Similarity Scores
how closely chunks related to your qurey 
ChromaDb uses L2 distance (Euclidean distance)

-Lower Scores = more similar
-score of 0 = identical
-range 0 to 2 (but can be higher)

Cosine similarity:
-higher Scores =  More similar
-Range -1 to 1 ( 1 = identical)

### Initailizign LLM, RAG Chain, Promt template, Query the RAG system

In [96]:
from dotenv import load_dotenv
load_dotenv()

True

In [97]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b")

In [98]:
print(llm.invoke("Say hello in one line").content)

Hello


In [99]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("groq:openai/gpt-oss-120b")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B78592AAB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B7854E76B0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [100]:
llm.invoke("what is ai")

AIMessage(content='**Artificial Intelligence (AI)** is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks can include learning, reasoning, problem‑solving, perception, language understanding, and even creativity.\n\n### Core Ideas\n\n| Concept | What It Means |\n|---------|---------------|\n| **Machine Learning (ML)** | Algorithms that improve their performance automatically as they are exposed to more data. |\n| **Deep Learning** | A subset of ML that uses multi‑layered neural networks to model complex patterns (e.g., image and speech recognition). |\n| **Natural Language Processing (NLP)** | Techniques for understanding, generating, and translating human language. |\n| **Computer Vision** | Enabling machines to interpret visual information from images or video. |\n| **Reinforcement Learning** | Training agents to make sequences of decisions by rewarding desirable outcomes. |\n| **Reasoning & 

### Modern RAG chain

In [101]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001B7640151F0>, search_kwargs={'k': 3})

In [102]:
## Create a prompt template for the retrieval chain
## context here is the
from langchain_core.prompts import ChatPromptTemplate 

system_prompt = """ You are a helpful assistant that answers questions based on the context provided. 
If the answer is not contained within the context, respond with "I don't know." 
Use the following pieces of context to answer the question at the end.

Context: {context}   
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [103]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' You are a helpful assistant that answers questions based on the context provided. \nIf the answer is not contained within the context, respond with "I don\'t know." \nUse the following pieces of context to answer the question at the end.\n\nContext: {context}   \n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [104]:
## Create a Document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' You are a helpful assistant that answers questions based on the context provided. \nIf the answer is not contained within the context, respond with "I don\'t know." \nUse the following pieces of context to answer the question at the end.\n\nContext: {context}   \n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_input

## this chain:
- Takes retreieved documents ( chuncks )
- "Stuffs" them into the prompt {context}
- sends complete prompt to the LLM 
- Returns the LLM response

In [105]:
## Creating the final rag chain

from langchain_classic.chains import create_retrieval_chain
rag_chain = create_retrieval_chain(retriever, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001B7640151F0>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' You are a helpful assistant that answers questions based on the context provided. \nIf the answer is not contained within th

In [106]:
response = rag_chain.invoke({"input": "What is deep learning ?"})
response['answer']

'Deep learning is a specialized subset of machine learning that employs artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of data. Inspired by the structure and functioning of the human brain, these models consist of interconnected artificial neurons organized into input, hidden, and output layers. Each layer processes information and extracts increasingly complex features, enabling the model to perform sophisticated tasks.'

### Create RAG Chain Using LCEL(LangChain Expression Languague)

In [107]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnableMap, RunnableLambda, RunnablePassthrough, Runnable



In [108]:
#creating a custom prompt 

custom_prompt = ChatPromptTemplate.from_template(
    """
    Use the following context to answer the question. If the answer is not contained within the context, respond with "I don't know."
    context: {context}
    question: {question}
    Answer:
    """
)
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n    Use the following context to answer the question. If the answer is not contained within the context, respond with "I don\'t know."\n    context: {context}\n    question: {question}\n    Answer:\n    '), additional_kwargs={})])

In [109]:
# Format the output docs 
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [110]:
## BUILD the rag chain using lcel

rag_chain_lcel = (
    {"context": retriever | format_docs ,
     "question": RunnablePassthrough() }
     | custom_prompt
     | llm
     | StrOutputParser()
)

rag_chain_lcel


{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001B7640151F0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n    Use the following context to answer the question. If the answer is not contained within the context, respond with "I don\'t know."\n    context: {context}\n    question: {question}\n    Answer:\n    '), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_c

In [111]:
response_lcel = rag_chain_lcel.invoke("What is deep learning ?")

response_lcel

'Deep learning is a specialized subset of machine learning that employs artificial neural networks with multiple (input, hidden, and output) layers to automatically learn complex patterns and representations from large amounts of data. By stacking many layers, these models can extract increasingly abstract features and perform sophisticated tasks such as image, speech, and language recognition.'

In [112]:
## chunks
docs = retriever.invoke("what is deep learning?")

for d in docs:
    print(d.metadata["source"])
    print(d.page_content[:200])
    print("---")

data\doc_1.txt
Deep Learning
Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of da
---
data\doc_1.txt
Deep Learning
Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of da
---
data\doc_1.txt
Deep Learning
Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns and representations from large amounts of da
---


## Adding New Documents to the Existion Vector Store

In [113]:
vectorstore

In [114]:
## adding new doc

new_document = """
Reinforcement Learning (RL) is a branch of Machine Learning that focuses on training agents to make sequences of decisions by interacting with an environment. In RL, an agent learns to achieve a goal by taking actions and receiving feedback in the form of rewards or penalties. The agent's objective is to maximize the cumulative reward over time. RL is inspired by behavioral psychology and is used in various applications, including robotics, game playing, autonomous vehicles, and recommendation systems. Key concepts in RL include states, actions, rewards, policies, value functions, and exploration-exploitation trade-offs. Popular algorithms in RL include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and Actor-Critic methods. RL has gained significant attention due to its success in complex tasks such as playing Go, chess, and video games at superhuman levels.                    
"""

new_document

"\nReinforcement Learning (RL) is a branch of Machine Learning that focuses on training agents to make sequences of decisions by interacting with an environment. In RL, an agent learns to achieve a goal by taking actions and receiving feedback in the form of rewards or penalties. The agent's objective is to maximize the cumulative reward over time. RL is inspired by behavioral psychology and is used in various applications, including robotics, game playing, autonomous vehicles, and recommendation systems. Key concepts in RL include states, actions, rewards, policies, value functions, and exploration-exploitation trade-offs. Popular algorithms in RL include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and Actor-Critic methods. RL has gained significant attention due to its success in complex tasks such as playing Go, chess, and video games at superhuman levels.                    \n"

In [ ]:
from langchain_core.documents import Document
new_doc = Document(``
    page_content=new_document, 
    metadata={"source": "new_doc.txt", "topic": "Reinforcement Learning"    }
)

In [117]:
new_doc

Document(metadata={'source': 'new_doc.txt', 'topic': 'Reinforcement Learning'}, page_content="\nReinforcement Learning (RL) is a branch of Machine Learning that focuses on training agents to make sequences of decisions by interacting with an environment. In RL, an agent learns to achieve a goal by taking actions and receiving feedback in the form of rewards or penalties. The agent's objective is to maximize the cumulative reward over time. RL is inspired by behavioral psychology and is used in various applications, including robotics, game playing, autonomous vehicles, and recommendation systems. Key concepts in RL include states, actions, rewards, policies, value functions, and exploration-exploitation trade-offs. Popular algorithms in RL include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and Actor-Critic methods. RL has gained significant attention due to its success in complex tasks such as playing Go, chess, and video games at superhuman levels.                    

In [118]:
new_chunks = text_splitter.split_documents([new_doc])

In [119]:
new_chunks

[Document(metadata={'source': 'new_doc.txt', 'topic': 'Reinforcement Learning'}, page_content="Reinforcement Learning (RL) is a branch of Machine Learning that focuses on training agents to make sequences of decisions by interacting with an environment. In RL, an agent learns to achieve a goal by taking actions and receiving feedback in the form of rewards or penalties. The agent's objective is to maximize the cumulative reward over time. RL is inspired by behavioral psychology and is used in various applications, including robotics, game playing, autonomous vehicles, and recommendation"),
 Document(metadata={'source': 'new_doc.txt', 'topic': 'Reinforcement Learning'}, page_content='playing, autonomous vehicles, and recommendation systems. Key concepts in RL include states, actions, rewards, policies, value functions, and exploration-exploitation trade-offs. Popular algorithms in RL include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and Actor-Critic methods. RL has gai

In [120]:
## adding new chunks to the vectorstore

vectorstore.add_documents(new_chunks)

['6b575e9f-e488-4e9b-8cde-95d9a4249b2c',
 '7ad12ae4-2680-4a0f-9ebe-c484b9967b12']

In [121]:
print(f"Number of documents in the vector store after adding new chunks: {vectorstore._collection.count()}")

Number of documents in the vector store after adding new chunks: 57


### Advanced RAG techniques : Conversational Memory

makes the retriever understand conversation context ( like before messages)
placeholder for chat history in prompts
message types for conversation history

In [123]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage


In [124]:
### Creating a prompt that includes chat history
contextualize_q_system_prompt = """ You are a helpful assistant that answers questions based on the context provided.
If the answer is not contained within the context, respond with "I don't know." do not make up answers.
"""

contextualize_system_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    ("human", "{input}"),
    (MessagesPlaceholder(variable_name="chat_history"))
])